<a href="https://colab.research.google.com/github/Gcarmnonapy7/FIAP-Aurora-Siger/blob/main/Aurora_siger.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Pipeline de Machine Learning

```
DATA LOADING & CLEANING → EDA → FEATURE ENGINEERING → DATA SPLITTING → TRAINING → VALIDATION ⇄ HYPERPARAMETER TUNING → SAVE ARTEFACT
```

> **Nota:** Validação e Hyperparameter Tuning formam um ciclo iterativo — ajusta-se, valida-se, repete-se até encontrar a melhor configuração.

| Etapa | O que faz |
|-------|-----------|
| **Data Loading & Cleaning** | Carregar os dados brutos e tratar valores ausentes, duplicados ou inconsistentes |
| **EDA** (Exploratory Data Analysis) | Visualizar distribuições, correlações e padrões para entender os dados antes de modelar |
| **Feature Engineering** | Criar, transformar ou selecionar variáveis que melhorem a capacidade preditiva do modelo |
| **Data Splitting** | Dividir em treino, validação e teste para avaliar o modelo em dados que ele nunca viu |
| **Training** | Ajustar os parâmetros do modelo aos dados de treino |
| **Validation** | Medir o desempenho em dados de validação para detectar overfitting ou underfitting |
| **Hyperparameter Tuning** | Otimizar configurações externas ao modelo (ex: número de árvores, kernel) que não são aprendidas no treino |
| **Save Artefact** | Serializar o modelo treinado (ex: `joblib`, `pickle`) para uso em produção |

---

# Entregáveis do Projeto Aurora SIGER

Roteiro de desenvolvimento do trabalho em grupo. Cada seção abaixo corresponde a um entregável da atividade.

## 1.1 Organização e descrição da telemetria

Interpretar dados referentes a:
- Temperatura interna e externa
- Integridade estrutural (0/1)
- Níveis de energia (%)
- Pressão dos tanques
- Status dos módulos críticos

In [1]:
%pip install -q optuna

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 419.5/419.5 kB 11.3 MB/s eta 0:00:00


In [2]:
# === import libraries ===

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.svm import OneClassSVM
from sklearn.ensemble import IsolationForest

In [3]:
np.random.seed(42)
n_samples = 1_000_000
anomalies_ratio = 0.03
n_anomalies = int(n_samples * anomalies_ratio)

In [4]:
tank_pressure_data = np.random.normal(loc=305, scale=15, size=n_samples - n_anomalies)

failure_prob = 1 / (1 + np.exp(-(tank_pressure_data - 340) / 5))
failure_prob = np.clip(failure_prob, 0, 1)

df_non_anomaly = pd.DataFrame({
    'internal_temp': np.random.normal(loc=22, scale=1.5, size=n_samples - n_anomalies),
    'external_temp': np.random.normal(loc=10, scale=8, size=n_samples - n_anomalies),
    'structural_integrity': np.random.binomial(1, 1 - failure_prob),
    'energy': np.clip(np.random.normal(loc=98, scale=2, size=n_samples - n_anomalies), 0, 100),
    'vibration': np.random.normal(loc=0.3, scale=0.1, size=n_samples - n_anomalies),
    'tank_pressure': tank_pressure_data,
    'critical_modules': np.random.binomial(1, 1 - failure_prob, size=n_samples - n_anomalies)
})

In [5]:
# --- ANOMALY PRESSURE (shift distribution) ---
tank_pressure_anomaly = np.random.normal(loc=360, scale=25, size=n_anomalies)

# Higher failure probability (more aggressive sigmoid)
failure_prob_anomaly = 1 / (1 + np.exp(-(tank_pressure_anomaly - 300) / 5))
failure_prob_anomaly = np.clip(failure_prob_anomaly, 0, 1)

internal_temp =  np.random.choice(np.concatenate(
       [np.random.normal(35,3,size=n_anomalies//2),
        np.random.normal(5,2,size=n_anomalies//2)] # need to divide it for the right number
    )) # internal temp (extreme)

structural_integrity = np.clip(np.random.binomial(
    1,
    0.3 + 0.4*(1 - failure_prob_anomaly)
),0,1)

df_anomaly = pd.DataFrame({
    'internal_temp': internal_temp,
    'external_temp': np.random.normal(loc=60, scale=20, size=n_anomalies),
    'structural_integrity': np.random.binomial(1, 1 - failure_prob_anomaly),
    'energy': np.clip(np.random.normal(loc=40, scale=15, size=n_anomalies), 0, 100),
    'vibration': np.random.normal(loc=1.2, scale=0.4, size=n_anomalies),
    'critical_modules': np.random.binomial(1,failure_prob_anomaly, size=n_anomalies)})


# === anomaly columns ===

df_non_anomaly['anomaly'] = 0
df_anomaly['anomaly'] = 1

init_dataframe = pd.concat([df_non_anomaly,df_anomaly]).reset_index(drop=True)

In [6]:
print(f'{init_dataframe.shape}')

(1000000, 8)


In [7]:
init_dataframe.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000000 entries, 0 to 999999
Data columns (total 8 columns):
 #   Column                Non-Null Count    Dtype  
---  ------                --------------    -----  
 0   internal_temp         1000000 non-null  float64
 1   external_temp         1000000 non-null  float64
 2   structural_integrity  1000000 non-null  int64  
 3   energy                1000000 non-null  float64
 4   vibration             1000000 non-null  float64
 5   tank_pressure         970000 non-null   float64
 6   critical_modules      1000000 non-null  int64  
 7   anomaly               1000000 non-null  int64  
dtypes: float64(5), int64(3)
memory usage: 61.0 MB


In [8]:
init_dataframe.describe()

,internal_temp,external_temp,structural_integrity,energy,vibration,tank_pressure,critical_modules,anomaly
count,1000000.000000,1000000.000000,1000000.000000,1000000.000000,1000000.000000,970000.000000,1000000.000000,1000000.000000
mean,21.458388,11.507476,0.947997,96.103623,0.326983,304.979112,0.977256,0.030000
std,3.411235,12.143583,0.222033,10.319150,0.195070,15.006311,0.149086,0.170587
min,3.979234,-29.841167,0.000000,0.000000,-0.272418,232.558460,0.000000,0.000000
25%,20.873493,4.800694,1.000000,96.501975,0.234858,294.846572,1.000000,0.000000
50%,21.941354,10.320238,1.000000,97.925887,0.303644,304.987418,1.000000,0.000000
75%,22.974493,15.984088,1.000000,99.301727,0.374866,315.106274,1.000000,0.000000
max,29.241434,140.368337,1.000000,100.000000,2.633571,375.184237,1.000000,1.000000


## EDA (Exploratory data analysis)

In [9]:
import seaborn as sns
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
import plotly.express as px

# === Plot configurations ===
sns.set_style('darkgrid')
sns.set_context('talk') # bold text for readable
sns.despine() # remove the top and right spines

def heatmap_plot(data:pd.DataFrame) -> None:
    plt.figure(figsize=(7.5,6))
    plt.title('Heatmap of the correlation between variables')
    corr = data.corr()
    mask = np.triu(np.ones_like(corr, dtype=bool)) # upper triangular (down visualization) for easier visualization
    sns.heatmap(corr,mask=mask,annot=True,fmt=".2f")
    plt.show()

def distribution_plot(data: pd.DataFrame) -> None:

    n_vars = len(data.columns)
    n_cols = 2
    n_rows = (n_vars + n_cols - 1) // n_cols

    plt.figure(figsize=(16, 4*n_rows))

    for idx, col in enumerate(data.columns, start=1):

        plt.subplot(n_rows, n_cols, idx)

        sns.histplot(
            data[col],
            kde=True,
            bins=25,
            color="#4C72B0",
            edgecolor="black"
        )

        plt.title(f"Distribution of {col}", fontsize=12, fontweight="bold")
        plt.xlabel(col)
        plt.ylabel("Frequency")

    plt.suptitle("Feature Distributions", fontsize=18, fontweight="bold")

    plt.tight_layout(rect=[0,0,1,0.96])
    plt.show()

def boxplot_(data:pd.DataFrame):

  n_vars = len(data.columns)
  n_cols = 3
  n_rows = (n_vars + n_cols -1) // n_cols

  plt.figure(figsize=(16,4*n_rows))
  for idx, col in enumerate(data.columns):

        plt.subplot(n_rows, n_cols, idx + 1)

        sns.boxenplot(
            x=data[col],
            color="#4C72B0"
        )

        plt.title(f"{col}", fontsize=12, fontweight="bold")

  plt.suptitle("Feature Distribution Overview", fontsize=18, fontweight="bold")

  plt.tight_layout()
  plt.show()

def statistical_analysis(data:pd.DataFrame):

    valid_columns = list(data.columns)
    for col in data.columns:
      print(f"{col}")
      print("*" * 60)

      print("*" * 60)

<Figure size 640x480 with 0 Axes>

In [10]:
def scattering_data(data: pd.DataFrame):
    if data is None or data.empty:
        print("Dataframe is empty")
        return

    plot_cols = [c for c in data.columns if c != 'anomaly']
    n_vars = len(plot_cols)
    n_cols = 2
    n_rows = (n_vars + n_cols - 1) // n_cols

    fig, axes = plt.subplots(n_rows, n_cols, figsize=(16, n_rows * 4))
    axes = axes.flatten() # Flatten to iterate easily

    for idx, col in enumerate(plot_cols):
        sns.scatterplot(
            data=data,
            x=data.index,
            y=col,
            hue="anomaly",
            palette={0: "#4c72b0", 1: "#dd8452"},
            alpha=0.6,
            ax=axes[idx]  # Explicitly tell seaborn which subplot to use
        )
        axes[idx].set_title(col)
        axes[idx].set_xlabel("Index")

    # Remove any unused subplots (if n_vars is odd)
    for i in range(idx + 1, len(axes)):
        fig.delaxes(axes[i])

    plt.suptitle("Feature Scatter Overview", fontsize=18, fontweight="bold")
    plt.tight_layout(rect=[0, 0.03, 1, 0.95]) # Adjust for suptitle
    plt.show()

In [ ]:
scattering_data(init_dataframe)

In [ ]:
import plotly.io as pio
pio.renderers.default = "colab"

def visualization_3D_anomaly(data: pd.DataFrame, x_axis: str, y_axis: str, z_axis: str):

    if data is None or data.empty:
        print("Dataframe is empty.")
        return

    required_cols = [x_axis, y_axis, z_axis, "anomaly"]

    for col in required_cols:
        if col not in data.columns:
            print(f"Column '{col}' not found in dataframe.")
            return

    fig = px.scatter_3d(
        data,
        x=x_axis,
        y=y_axis,
        z=z_axis,
        color="anomaly",
        opacity=0.7,
        title=f"3D Anomaly Visualization: {x_axis} vs {y_axis} vs {z_axis} (Partialy...)"
    )

    fig.update_layout(
        scene=dict(
            xaxis_title=x_axis,
            yaxis_title=y_axis,
            zaxis_title=z_axis
        ),
        width=900,
        height=700,
        legend_title="Anomaly"
    )

    fig.show()


normal = init_dataframe[init_dataframe["anomaly"] == 0].sample(3000) # 9000 samples for plotting
anomaly = init_dataframe[init_dataframe["anomaly"] == 1]

plot_df = pd.concat([normal,anomaly],axis=0)

visualization_3D_anomaly(
    plot_df,
    "tank_pressure",
    "internal_temp",
    "vibration"
)

In [ ]:
heatmap_plot(init_dataframe) # not so much correlated (only tank and pressure)

In [ ]:
init_dataframe.head()

In [ ]:
## exploring the data
plt.title("Value counts about structural_integrity")
init_dataframe['structural_integrity'].value_counts().plot(kind="bar")
plt.tight_layout()
plt.show()

In [ ]:
boxplot_(data=init_dataframe)

In [ ]:
distribution_plot(data=init_dataframe)

##Creating the models from scratch

In [ ]:
# === Decision tree from scratch ===

class Node:
  def __init__(self,feature,threshold,left=None,right=None):

    self.feature = feature
    self.threshold =threshold



class DecisionTree:
  def __init__(self):
    pass


In [ ]:
# Isolation tree from strach
class IsolationTree:
  def __init__(self):
    pass

In [ ]:
def verifying_metrics():
  pass

## 1.2 Algoritmo de verificação

Construir um algoritmo (fluxograma/pseudocódigo) capaz de decidir:
- **"PRONTO PARA DECOLAR"** ou **"DECOLAGEM ABORTADA"**

Com base em faixas seguras predefinidas para cada variável de telemetria.

```
algoritmo "Verificação de Decolagem Aurora SIGER"

variáveis

    internal_temp, external_temp: real

    structural_integrity, critical_modules: inteiro

    energy: real

    vibration: real

    tank_pressure: real

    abortado: lógico

início

    abortado = FALSO

    Escreva "=== SISTEMA AURORA SIGER — VERIFICAÇÃO PRÉ-DECOLAGEM ==="

    Escreva "Lendo dados de telemetria..."

    Leia internal_temp

    Leia external_temp

    Leia structural_integrity

    Leia energy

    Leia vibration

    Leia tank_pressure

    Leia critical_modules

    // Verificação da temperatura interna (faixa ISS: 18–26 °C)

    Se internal_temp < 18 OU internal_temp > 26 então

        Escreva "[ALERTA] Temperatura interna fora da faixa segura: ", internal_temp, " °C"

        abortado = VERDADEIRO

    Fim_se

    // Verificação da temperatura externa (faixa LEO: -65 a +125 °C)

    Se external_temp < -65 OU external_temp > 125 então

        Escreva "[ALERTA] Temperatura externa fora da faixa segura: ", external_temp, " °C"

        abortado = VERDADEIRO

    Fim_se

    // Verificação da integridade estrutural (1 = íntegro, 0 = falha)

    Se structural_integrity = 0 então

        Escreva "[ALERTA] Falha na integridade estrutural detectada"

        abortado = VERDADEIRO

    Fim_se

    // Verificação do nível de energia (mínimo Go/No-Go: 95%)

    Se energy < 95 então

        Escreva "[ALERTA] Nível de energia insuficiente: ", energy, " %"

        abortado = VERDADEIRO

    Fim_se

    // Verificação da vibração (faixa segura pré-decolagem: 0.1–0.5 g)

    Se vibration < 0.1 OU vibration > 0.5 então

        Escreva "[ALERTA] Vibração fora da faixa segura: ", vibration, " g"

        abortado = VERDADEIRO

    Fim_se

    // Verificação da pressão dos tanques (faixa segura: 270–340 atm)

    Se tank_pressure < 270 OU tank_pressure > 340 então

        Escreva "[ALERTA] Pressão dos tanques fora da faixa segura: ", tank_pressure, " atm"

        abortado = VERDADEIRO

    Fim_se

    // Verificação do status dos módulos críticos (1 = ativo, 0 = inativo)

    Se critical_modules = 0 então

        Escreva "[ALERTA] Módulos críticos inativos"

        abortado = VERDADEIRO

    Fim_se

    // Decisão final

    Se abortado = VERDADEIRO então

        Escreva ">>> DECOLAGEM ABORTADA <<<"

    Fim_se

    Se abortado = FALSO então

        Escreva ">>> PRONTO PARA DECOLAR <<<"

    Fim_se

Fim
```

### Fluxograma do Algoritmo de Verificação

![Fluxograma de Verificação de Decolagem](https://github.com/Gcarmnonapy7/FIAP-Aurora-Siger/blob/main/assets/fluxograma_verificacao.png?raw=1)

## 1.3 Script em Python

Implementar a lógica do algoritmo em Python, simulando:
- Leitura dos dados de telemetria
- Execução das verificações contra faixas seguras
- Resultado final impresso: **"PRONTO PARA DECOLAR"** ou **"DECOLAGEM ABORTADA"**

In [ ]:
# === 1.3 Script de Verificação Pré-Decolagem ===
# Tradução direta do pseudocódigo (1.2) para Python

# Faixas seguras (mesmos valores do pseudocódigo e fluxograma)
SAFE_RANGES = {
    'internal_temp': (18, 26),              # °C — ISS: 18–26°C
    'external_temp': (-65, 125),            # °C — LEO: -65 a +125°C
    'structural_integrity': (1, 1),         # 1 = íntegro, 0 = falha (só 1 é aceitável)
    'energy': (95, 100),                    # % — Go/No-Go ≥95%
    'vibration': (0.1, 0.5),               # g — pré-decolagem
    'tank_pressure': (270, 340),           # atm — LOX/LH2 pump-fed
    'critical_modules': (1, 1),            # 1 = ativo (só 1 é aceitável)
}

def check_launch_readiness(reading: dict) -> bool:
    """
    Recebe uma leitura de telemetria (dict ou pd.Series) e verifica
    se todos os parâmetros estão dentro das faixas seguras.

    Retorna True se PRONTO PARA DECOLAR, False se DECOLAGEM ABORTADA.
    """
    print("=== SISTEMA AURORA SIGER — VERIFICAÇÃO PRÉ-DECOLAGEM ===")
    print("Lendo dados de telemetria...\n")

    aborted = False

    for var, (min_val, max_val) in SAFE_RANGES.items():
        if reading[var] >= min_val and reading[var] <= max_val:
            print(f"{var}, com valor {reading[var]} está dentro da faixa segura.")

        if reading[var] < min_val or reading[var] > max_val:
            aborted = True
            print(f"[ALERTA]: {var}, com valor {reading[var]} está fora da faixa segura!")

    # Decisão final
    if aborted:
        print("\n>>> DECOLAGEM ABORTADA <<<")
    else:
        print("\n>>> PRONTO PARA DECOLAR <<<")

    return not aborted

# --- Teste com uma leitura do DataFrame ---
sample_reading = df_non_anomaly.iloc[0].to_dict()
print(f"Leitura de teste: {sample_reading}\n")
result = check_launch_readiness(sample_reading)

sample_reading = df_non_anomaly.iloc[2].to_dict()
print(f"Leitura de teste: {sample_reading}\n")
result = check_launch_readiness(sample_reading)

## 1.4 Análise energética

Calcular autonomia inicial considerando:
- Capacidade total (kWh)
- Carga atual (%)
- Consumo estimado na decolagem
- Perdas energéticas

### Considerações sobre o gerenciamento energético em foguetes espaciais

O gerenciamento energético de uma missão espacial é um exercício de engenharia de precisão onde cada quilowatt-hora conta. A capacidade total simulada neste projeto — 18 kWh, cobrindo cápsula e estágio superior — é comparável à de um carro elétrico urbano de entrada de gama. A diferença fundamental, porém, está no contexto: enquanto um veículo terrestre pode ser recarregado em qualquer ponto da rede elétrica, uma cápsula em órbita depende exclusivamente da energia embarcada até que os painéis solares sejam desdobrados e estabilizados. Essa limitação torna imperativo que sistemas como o Aurora SIGER monitorizem a carga em tempo real e imponham limiares rígidos de Go/No-Go — no nosso modelo, 95 % de carga pré-lançamento, alinhado com os ≥ 97 % exigidos em missões tripuladas reais.

Alcançar níveis elevados de eficiência energética no espaço esbarra em restrições físicas severas. Baterias de lítio-íon, mesmo as de grau espacial, sofrem perdas por resistência interna (2–5 %) que se agravam com as variações térmicas extremas entre face solar e sombra orbital. A cadeia de conversão DC-DC, responsável por alimentar barramentos de 5 V e 12 V a partir do barramento principal de 28 VDC (MIL-STD-704F), consome entre 3 % e 8 % da energia. Somam-se as perdas em chicotes elétricos que podem ultrapassar 100 metros de comprimento (1–3 %), os sistemas de gestão térmica das próprias baterias (2–5 %) e o condicionamento de potência com reguladores e filtros de proteção (1–2 %). No total, as perdas do nosso modelo atingem 14 %, reduzindo a autonomia orbital estimada para cerca de 12 horas — uma janela que, embora suficiente para manobras iniciais, exige planeamento rigoroso de cada subsistema consumidor.

Essas restrições tornam-se ainda mais relevantes à luz dos planos da NASA de estabelecer presença humana contínua na Lua a partir de 2027, no âmbito do programa Artemis. Missões de longa duração demandarão avanços significativos em armazenamento energético, sistemas de recarga in situ e eficiência de conversão. A sustentabilidade da exploração espacial dependerá, em última análise, da capacidade de minimizar perdas e maximizar a autonomia com recursos finitos — precisamente o tipo de análise que ferramentas de telemetria como o Aurora SIGER pretendem viabilizar.

> **Fontes:** As referências completas dos parâmetros energéticos (SpaceX Crew Dragon Press Kit, NASA Orion MPCV, MIL-STD-704F, *Space Mission Engineering: The New SMAD*) encontram-se na secção **"Parâmetros energéticos"** do [README.md](README.md).

In [ ]:
# === Parâmetros energéticos (baseados em sistemas reais) ===
# Fontes: SpaceX Crew Dragon Press Kit, NASA Orion MPCV, MIL-STD-704F,
#         "Space Mission Engineering: The New SMAD" (Wertz et al., Cap. 11)

# --- Capacidade e carga ---
total_energy_capacity_kwh = 18.0    # Cápsula + estágio superior (Crew Dragon: 10–20 kWh, Orion: ~11 kWh)
pre_launch_charge_pct = 100.0       # Baterias carregadas via GSE até desconexão do umbilical
min_launch_charge_pct = 95.0        # Limiar Go/No-Go (missões tripuladas: ≥97%)

# --- Consumo na fase de lançamento ---
launch_power_draw_kw = 2.0          # Consumo contínuo: aviônica + GNC + telemetria + atuadores (1.5–3.0 kW)
launch_duration_min = 9.0           # Tempo até inserção orbital (~8–10 min)
peak_power_kw = 4.0                 # Picos breves durante staging (3–5 kW) — referência, não entra no cálculo médio

# --- Consumo orbital (pós-lançamento) ---
orbital_power_draw_kw = 1.2         # Sem atuadores/pirotécnicos, GNC em cruzeiro (Crew Dragon: ~1.0–1.2 kW)

# --- Perdas energéticas (total ~8–18%, média 14%) ---
battery_resistance_loss_pct = 3.0   # Resistência interna Li-ion (I²R): 2–5%
dc_dc_conversion_loss_pct = 5.0     # Conversores para barramentos 5V, 12V: 3–8%
wiring_loss_pct = 2.0               # Chicotes elétricos 100+ metros: 1–3%
thermal_management_loss_pct = 3.0   # Aquecedores de bateria + refrigeração: 2–5%
power_conditioning_loss_pct = 1.0   # Reguladores, filtros, proteção: 1–2%

total_loss_pct = (battery_resistance_loss_pct
                + dc_dc_conversion_loss_pct
                + wiring_loss_pct
                + thermal_management_loss_pct
                + power_conditioning_loss_pct)

# --- Resumo dos parâmetros ---
print(f"Capacidade total: {total_energy_capacity_kwh} kWh")
print(f"Carga pré-lançamento: {pre_launch_charge_pct}%")
print(f"Consumo lançamento: {launch_power_draw_kw} kW × {launch_duration_min} min")
print(f"Consumo orbital: {orbital_power_draw_kw} kW (~{orbital_power_draw_kw/launch_power_draw_kw*100:.0f}% do lançamento)")
print(f"Perdas totais: {total_loss_pct}%")
print(f"Barramento padrão: 28 VDC (MIL-STD-704F)")


def calculate_autonomy(capacity_kwh: float,
                       charge_pct: float,
                       loss_pct: float,
                       launch_draw_kw: float,
                       launch_min: float,
                       orbital_draw_kw: float,
                       min_charge_pct: float = 95.0) -> float | None:
    """
    Calcula a autonomia estimada (horas) em órbita após a fase de lançamento.

    Fórmula:
        energia_útil       = capacity × charge% × (1 - loss%)
        energia_lançamento = launch_draw × launch_min / 60
        autonomia          = (energia_útil - energia_lançamento) / orbital_draw

    Retorna None se a carga pré-lançamento estiver abaixo do limiar Go/No-Go.
    """
    # Verificação Go/No-Go: carga mínima exigida antes da decolagem
    if charge_pct < min_charge_pct:
        print(f"[ABORTADO] Carga pré-lançamento ({charge_pct}%) "
              f"abaixo do mínimo exigido ({min_charge_pct}%).")
        return None

    # Energia disponível = capacidade × carga × eficiência (η = 1 - perdas)
    available_energy_kwh = capacity_kwh * (charge_pct / 100) * (1 - loss_pct / 100)

    # Energia consumida durante o lançamento (conversão min → h)
    launch_energy_kwh = launch_draw_kw * (launch_min / 60)

    # Autonomia orbital = energia restante / consumo em cruzeiro
    autonomy_hr = (available_energy_kwh - launch_energy_kwh) / orbital_draw_kw

    return autonomy_hr


# --- Cálculo final ---
autonomy = calculate_autonomy(
    capacity_kwh=total_energy_capacity_kwh,
    charge_pct=pre_launch_charge_pct,
    loss_pct=total_loss_pct,
    launch_draw_kw=launch_power_draw_kw,
    launch_min=launch_duration_min,
    orbital_draw_kw=orbital_power_draw_kw,
    min_charge_pct=min_launch_charge_pct,
)

if autonomy is not None:
    print(f"\nAutonomia estimada em órbita: {autonomy:.2f} horas")

## 1.5 Análise assistida por IA

Solicitar à IA:
- Classificação dos dados de telemetria
- Identificação de possíveis anomalias
- Sugestões de risco

## 1.6 Reflexão crítica

Texto sobre:
- Ética e responsabilidade no uso de IA e automação em sistemas críticos
- Impacto social da exploração espacial
- Sustentabilidade tecnológica

# Quem decide quando a máquina decide?

*Sobre ética, automação e os limites do progresso — do algoritmo que nega seu crédito ao satélite que monitora o clima."*

---

Você abre o celular e o feed já está montado — alguém decidiu o que você veria. Você pesquisa um voo e o preço muda na próxima visita — alguém decidiu quanto você pagaria. Você manda um currículo e ele desaparece antes de qualquer humano ler — alguém decidiu que você não era relevante. Só que esse "alguém" não é uma pessoa. É um algoritmo — um conjunto de regras escritas por um programador, seguindo critérios definidos por uma empresa. A lógica está ali, mas a *intenção* por trás dela — por que esses critérios e não outros, a quem servem, o que otimizam — ficou na cabeça de quem projetou o sistema. Não há obrigação de torná-la visível. E é exatamente aí que mora o problema: as intenções embutidas em cada camada da cadeia tecnológica moldam silenciosamente o mundo que estamos construindo.

Não por acaso, esses modelos já foram chamados de "armas de destruição matemática" ([O'Neil, 2016](https://www.penguinrandomhouse.com/books/241363/weapons-of-math-destruction-by-cathy-oneil/)). São opacos, escaláveis e codificam preconceitos que já existiam na sociedade — mas agora operam em velocidade industrial. Você é avaliado, classificado e filtrado dezenas de vezes por dia — sem saber os critérios, sem poder contestar, e muitas vezes sem nem perceber que isso está acontecendo.

O problema não é a automação em si. É a ausência de transparência, responsabilidade e governança sobre ela.

## Todo mundo concorda. Ninguém sabe como fazer.

Oitenta e quatro documentos de ética em inteligência artificial — de governos, empresas e instituições de pesquisa de dezenas de países — convergem em cinco princípios: transparência, justiça, não causar dano, responsabilidade e privacidade. Mas há divergência substancial sobre o que esses princípios significam na prática, quem deve implementá-los e como ([Jobin, Ienca & Vayena, 2019](https://www.nature.com/articles/s42256-019-0088-2)).

Quase todo mundo concorda que a IA deveria ser justa e transparente. Quase ninguém concorda sobre o que fazer para garantir isso.

Enquanto essa lacuna persiste, há uma dimensão do problema que recebe ainda menos atenção: o custo energético. Cada modelo treinado, cada algoritmo rodando em um *data center*, cada busca processada consome eletricidade. Segundo estimativas de 2019, as emissões de gases de efeito estufa relacionadas a tecnologias de informação e comunicação cresciam cerca de 9% ao ano, e metade dessas emissões vinham da fabricação dos equipamentos ([The Shift Project, 2019](https://theshiftproject.org/en/article/lean-ict-our-new-report/)). A eficiência energética não é um detalhe técnico — é uma questão ética. Cada watt desperdiçado por um sistema mal otimizado é um recurso natural que não volta.

E aqui surge uma tensão produtiva: a mesma tecnologia que consome energia massiva também pode ser usada para reduzi-la. Algoritmos de aprendizado de máquina já otimizam redes elétricas, preveem padrões de consumo e controlam a refrigeração de grandes centros de processamento de dados. Um exemplo: o *data center* do Google em Hamina, na Finlândia, usa água do mar gelada para resfriar seus servidores e inteligência artificial para ajustar o sistema em tempo real. O resultado é que a esmagadora maioria da energia consumida vai direto para o processamento — uma fração mínima é gasta com refrigeração e infraestrutura auxiliar. O que muda o jogo não é a tecnologia em si, mas a decisão de empregá-la para otimizar em vez de desperdiçar.

Essa distinção — entre uso que cria e uso que extrai — aparece também na pesquisa sobre cognição. Uma meta-análise com mais de 411 mil adultos demonstrou que o uso *ativo* de tecnologia digital (aprender, resolver problemas, manter conexões) reduz em 58% o risco de declínio cognitivo, um efeito protetor comparável ao do exercício físico. O uso passivo e compulsivo, por outro lado, está associado a déficits mensuráveis em atenção e função executiva ([Benge & Scullin, 2025](https://www.nature.com/articles/s41562-025-02159-9); [Throuvala et al., 2023](https://link.springer.com/article/10.1007/s11065-023-09612-4)). A mesma tela, o mesmo algoritmo — intenções diferentes, consequências opostas.

## Quando a fronteira reproduz a desigualdade

Se a ética da automação parece um problema abstrato, a exploração espacial oferece um caso concreto em que todas essas tensões se materializam.

Considere o programa Apollo. Custou mais de US$ 25 bilhões nos anos 1960 — aproximadamente US$ 260 bilhões em valores atuais. A economista Mariana Mazzucato argumenta que o retorno desse investimento superou amplamente o custo, e usa o Apollo como modelo para o tipo de inovação que precisamos: audaciosa, orientada por missões claras, com colaboração real entre Estado e setor privado ([Mazzucato, 2021](https://marianamazzucato.com/books/mission-economy)).

Os benefícios concretos são documentados: o programa espacial americano como um todo já transferiu mais de 2.000 tecnologias para uso civil — de sensores de câmera de celular a sistemas de purificação de água, de espuma viscoelástica a equipamentos de proteção para bombeiros ([NASA Spinoff](https://spinoff.nasa.gov/)). A exploração espacial não é luxo — é investimento com retorno mensurável.

Mas a pergunta que geralmente não se faz é: *quem* colhe esses retornos?

Se apenas países e empresas com capital massivo podem acessar recursos espaciais — mineração de asteroides, turismo orbital, satélites de comunicação —, o espaço reproduz no vácuo a mesma desigualdade que já existe na Terra. Joseph Pelton chama a corrida espacial privada de "nova corrida do ouro" — e aponta que, sem governança, seus benefícios se concentrarão ainda mais ([Pelton, 2017](https://link.springer.com/book/10.1007/978-3-319-39273-8)). O Tratado do Espaço Exterior de 1967 estabeleceu que o espaço é patrimônio de toda a humanidade e não pode ser reivindicado como território por nenhuma nação — mas a exploração comercial de seus recursos opera em zona cinzenta jurídica, e esse princípio está sendo testado como nunca.

O dilema é familiar: investimento público gera inovação transformadora, mas se a governança falha, os benefícios se privatizam e os custos se socializam. Exatamente como nos algoritmos que decidem sobre vidas sem prestar contas.

## A pergunta que atravessa tudo

Se há um fio que conecta a opacidade dos algoritmos, o consumo energético dos *data centers* e a corrida comercial pelo espaço, é este: **eficiência sem ética é apenas uma forma mais sofisticada de desperdício**.

Um algoritmo que otimiza lucro mas amplifica desigualdade não é eficiente — é destrutivo em escala. Um *data center* que processa bilhões de operações mas ignora sua pegada de carbono não é moderno — é insustentável. Uma indústria espacial que gera tecnologias transformadoras mas concentra seus benefícios não é inovadora — é extrativista.

E isso não é problema de especialista. É problema de qualquer pessoa que usa um celular, abre uma rede social ou paga um boleto — ou seja, de praticamente todo mundo. A exploração espacial, a regulamentação da IA e a eficiência energética parecem temas distantes, mas são, na verdade, questões sobre quem tem poder de decidir sobre a sua vida sem precisar pedir permissão.

A pergunta que vale a pena fazer — sobre automação, energia, espaço e tudo o mais — não é "essa tecnologia funciona?", mas "para quem funciona, com que transparência e sob qual governança?"

> A pergunta que vale a pena fazer sobre qualquer tecnologia não é "funciona?", mas "quem decidiu que deveria funcionar assim — e para quem?"
